In [0]:
import os
import time
import pyspark.pandas as ps

os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

from pyspark.sql import SparkSession
import pyspark.pandas as ps

spark = (
    SparkSession.builder
    .appName("HW4.10 Pandas API on Spark")
    .getOrCreate()
)

start = time.perf_counter()

df = ps.read_csv("orders.csv")

df["placed_at"] = ps.to_datetime(df["placed_at"])

df["revenue"] = df["quantity"] * df["price_each"]

df["month"] = df["placed_at"].dt.strftime("%Y-%m")

paid = df[df["status"] == "paid"]

monthly = (
    paid.groupby(["country", "month"])["revenue"]
    .sum()
    .reset_index()
    .sort_values(["country", "month"])
)

print(monthly.head(20))

end = time.perf_counter()

print(f"Elapsed time: {end - start:.4f} seconds")